# Lecture 10 — Transformers with a Physics Payoff
### (+ Evaluation / Data Pipeline)

**Course:** Machine Learning Applications in Physics (PHYG004)
**Block:** Foundations (Week 8, after presentation week)
**Data type:** Toy / Synthetic (names dataset + Ising spin chain)

This notebook has two sessions:

| Session | Content | Duration |
|---------|---------|----------|
| **Session A** | microgpt walkthrough (4 steps) + physics payoff segment | ~90 min |
| **Session B** | Evaluation, validation, and data-pipeline segment for term projects | ~90 min |

**Physics perspective**
- The Transformer is a sequence model, but its core operation — attention — is really
  a **data-dependent, input-adaptive many-body kernel**: $A_{ij} = \text{softmax}(q_i \cdot k_j / \sqrt{d})$.
  This contrasts with fixed-neighbourhood kernels (e.g. SchNet's radial cutoff).
- After building the model from scratch we visualize the attention matrix on a
  toy Ising spin chain to see this *dynamic neighbourhood selection* in action.
- We close with a structured evaluation / data-pipeline segment that you will
  apply directly to your term project.


## Setup

All microgpt cells use only the Python standard library (`os`, `math`, `random`).  
The physics-payoff and evaluation segments additionally use `matplotlib` and `numpy`.  
The optional HuggingFace appendix needs `transformers` and `datasets`.

In [ ]:
# Standard library only for microgpt — no pip install needed for Steps 1–4.
# Physics-payoff and evaluation segments:
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "matplotlib", "numpy"])
# Optional HuggingFace appendix (uncomment if you want to run Appendix B):
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
#                        "transformers", "datasets", "accelerate"])


In [ ]:
import os
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.dpi': 120})

random.seed(42)
np.random.seed(42)
print("Setup complete. Python stdlib + numpy + matplotlib loaded.")


---
## Session A — Building a GPT from Scratch (4 Steps)

We follow Andrej Karpathy's *microgpt* — a ~200-line pure Python GPT —
restructured into 4 substantive steps that each add one key idea:

| Step | New idea added |
|------|---------------|
| **Step 1** | MLP (manual forward + SGD), no attention |
| **Step 2** | Autograd (`Value` class), gradient check |
| **Step 3** | Single-head attention, position embeddings |
| **Step 4** | Multi-head attention + Adam optimizer |

**Note on Step 0 (bigram count baseline):** A bigram character model with no
neural network achieves ~2.9 bits/char on this dataset. You can run
`solution/step0_bigram_count.py` for reference. We start at Step 1 here.


### Step 1 — MLP (Manual Forward + SGD)

**New ideas:** token embeddings, a simple MLP with manual parameter initialization,
cross-entropy loss, SGD update.
No attention, no position information yet — the model sees only the current token.

**Physicist's note:**
The cross-entropy loss $\mathcal{L} = -\log p(\text{target})$ plays the role of a
*surprise* measure — how many bits does it take to encode the true next token
given the model's distribution? At random initialization the model assigns equal
probability $1/27$ to all tokens, giving $\mathcal{L}_0 = \log 27 \approx 3.30$.


In [ ]:
# ── Step 1: MLP with manual SGD ──────────────────────────────────────────────
if not os.path.exists('input.txt'):
    import urllib.request
    names_url = 'https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt'
    urllib.request.urlretrieve(names_url, 'input.txt')

docs_s1 = [l.strip() for l in open('input.txt').read().strip().split('\n') if l.strip()]
random.shuffle(docs_s1)
print(f"num docs: {len(docs_s1)}")

# Character-level tokenizer
uchars = sorted(set(''.join(docs_s1)))
BOS = len(uchars)
vocab_size = len(uchars) + 1
print(f"vocab size: {vocab_size}")
print(f"baseline loss (random): {math.log(vocab_size):.4f}")

# Hyperparameters
n_embd_s1 = 16

# Parameters (Python lists of floats — no autograd yet)
def make_matrix_s1(nout, nin, std=0.08):
    return [[random.gauss(0, std) for _ in range(nin)] for _ in range(nout)]

wte_s1  = make_matrix_s1(vocab_size, n_embd_s1)   # token embedding
w1_s1   = make_matrix_s1(4 * n_embd_s1, n_embd_s1)  # MLP layer 1
w2_s1   = make_matrix_s1(vocab_size, 4 * n_embd_s1) # MLP layer 2

params_s1 = [p for mat in [wte_s1, w1_s1, w2_s1] for row in mat for p in row]
print(f"num params (Step 1): {len(params_s1)}")
# ☑ Step 1 checkpoint: expected ~700 parameters


> **Step 1 checkpoint**
> - Number of parameters should be close to **~700** (embed: 27×16 = 432; MLP: 64×16 + 27×64 = 2752 — adjust dimensions as needed).
> - The model has no position information — it only sees the *current* token, not context.
> - After 1000 steps the loss should drop below the bigram baseline (~2.9).


### Step 2 — Autograd (`Value` class)

**New idea:** automatic differentiation via a scalar computation graph.
We implement the chain rule from scratch, then replace manual gradient calculations.

**Physicist's note:**
Autograd is the backbone of modern ML, but the idea is not new to physics.
The chain rule you learned in classical mechanics (Jacobian of composed maps)
is exactly what is applied here, one operation at a time. Each `Value` node stores
a local Jacobian scalar; backward() assembles them.


In [ ]:
# ── Step 2: Value class (scalar autograd) ────────────────────────────────────
class Value:
    """Scalar value with autograd. Tracks children + local derivatives."""
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0.0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other): return Value(self.data**other, (self,), (other * self.data**(other-1),))
    def log(self):  return Value(math.log(self.data),  (self,), (1/self.data,))
    def exp(self):  return Value(math.exp(self.data),  (self,), (math.exp(self.data),))
    def relu(self): return Value(max(0, self.data),    (self,), (float(self.data > 0),))
    def __neg__(self):          return self * -1
    def __radd__(self, other):  return self + other
    def __sub__(self, other):   return self + (-other)
    def __rsub__(self, other):  return other + (-self)
    def __rmul__(self, other):  return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

    def backward(self):
        """Run backprop: walk graph in reverse topological order, apply chain rule."""
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._children: build(c)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            for child, lg in zip(v._children, v._local_grads):
                child.grad += lg * v.grad

print("Value class defined.")


In [ ]:
# ── Gradient check ────────────────────────────────────────────────────────────
# Verify autograd against finite-difference approximation.
def finite_diff(f, x_val, eps=1e-5):
    """Central-difference gradient estimate for scalar input."""
    return (f(x_val + eps) - f(x_val - eps)) / (2 * eps)

# Test: f(a, b) = (a * b + a) ** 2 / b
a = Value(2.0)
b = Value(3.0)
c = (a * b + a) ** 2 / b
c.backward()

# Finite difference for da
fa = lambda av: ((av * 3.0 + av) ** 2) / 3.0
fa_fd = finite_diff(fa, 2.0)

# Finite difference for db
fb = lambda bv: ((2.0 * bv + 2.0) ** 2) / bv
fb_fd = finite_diff(fb, 3.0)

print(f"a.grad (autograd)     = {a.grad:.6f}")
print(f"a.grad (finite diff)  = {fa_fd:.6f}")
print(f"Relative error (a)    = {abs(a.grad - fa_fd)/abs(fa_fd + 1e-15):.2e}")
assert abs(a.grad - fa_fd) < 1e-4, "Autograd gradient mismatch!"

print(f"b.grad (autograd)     = {b.grad:.6f}")
print(f"b.grad (finite diff)  = {fb_fd:.6f}")
print(f"Relative error (b)    = {abs(b.grad - fb_fd)/abs(fb_fd + 1e-15):.2e}")
assert abs(b.grad - fb_fd) < 1e-4, "Autograd gradient mismatch!"
print("Gradient check PASSED (error < 1e-4).")


In [ ]:
# ── MLP with autograd ─────────────────────────────────────────────────────────
n_embd_s2 = 16

def make_matrix_v(nout, nin, std=0.08):
    return [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]

wte_s2  = make_matrix_v(vocab_size, n_embd_s2)
w1_s2   = make_matrix_v(4 * n_embd_s2, n_embd_s2)
w2_s2   = make_matrix_v(vocab_size, 4 * n_embd_s2)
params_s2 = [p for mat in [wte_s2, w1_s2, w2_s2] for row in mat for p in row]
print(f"num params (Step 2): {len(params_s2)}")

def linear_v(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

def softmax_v(logits):
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]

def mlp_s2(token_id):
    x = wte_s2[token_id]
    x = [xi.relu() for xi in linear_v(x, w1_s2)]
    return linear_v(x, w2_s2)

# Train Step 2: SGD
docs_s2 = docs_s1[:]  # same shuffled list
num_steps_s2 = 1000
learning_rate_s2 = 0.1
losses_s2_log = []
for step in range(num_steps_s2):
    doc = docs_s2[step % len(docs_s2)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    n = len(tokens) - 1
    losses_step = []
    for i in range(n):
        token_id, target_id = tokens[i], tokens[i+1]
        logits = mlp_s2(token_id)
        probs = softmax_v(logits)
        losses_step.append(-probs[target_id].log())
    loss_s2 = (1 / n) * sum(losses_step)
    loss_s2.backward()
    lr_t = learning_rate_s2 * (1 - step / num_steps_s2)
    for p in params_s2:
        p.data -= lr_t * p.grad
        p.grad = 0.0
    losses_s2_log.append(loss_s2.data)
    if step < 3 or (step + 1) % 200 == 0:
        print(f"step {step+1:4d}/{num_steps_s2} | loss {loss_s2.data:.4f}")


> **Step 2 checkpoint**
> - First step loss should be near **~3.6** (close to random init).
> - After 1000 steps loss should reach **~2.5** or lower.
> - Gradient check should pass (error < 1e-4 vs finite difference).
> - This model still sees only the current token — no context.


### Step 3 — Single-Head Attention

**New ideas:** position embeddings, queries / keys / values, scaled dot-product
attention (single head), RMSNorm, residual connections.

**Physicist's note:**
Attention computes a *data-dependent* weighted average of past token representations.
The weights $A_{ij} = \text{softmax}(q_i \cdot k_j / \sqrt{d})$ depend on the *content*
of the sequence, not just the positions — unlike fixed-kernel methods.

**Error #16 fix:** The attention scores are normalized by $\sqrt{d}$ where
$d = n\_embd$ for a **single-head** model. In Step 4 (multi-head), each head
operates on a slice of dimension $d_{\text{head}} = n\_embd / n\_head$, so the
scale becomes $\sqrt{d_{\text{head}}}$. Using $\sqrt{n\_embd}$ in the multi-head
code would be wrong and could cause training instability.


In [ ]:
# ── Step 3: Single-head attention ────────────────────────────────────────────
n_embd_s3 = 16
block_size = 16

def make_mat(nout, nin, std=0.08):
    return [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]

sd3 = {
    'wte':     make_mat(vocab_size, n_embd_s3),
    'wpe':     make_mat(block_size, n_embd_s3),
    'attn_wq': make_mat(n_embd_s3, n_embd_s3),
    'attn_wk': make_mat(n_embd_s3, n_embd_s3),
    'attn_wv': make_mat(n_embd_s3, n_embd_s3),
    'attn_wo': make_mat(n_embd_s3, n_embd_s3),
    'mlp_fc1': make_mat(4 * n_embd_s3, n_embd_s3),
    'mlp_fc2': make_mat(n_embd_s3, 4 * n_embd_s3),
    'lm_head': make_mat(vocab_size, n_embd_s3),
}
params_s3 = [p for mat in sd3.values() for row in mat for p in row]
print(f"num params (Step 3): {len(params_s3)}")

def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]

def gpt_s3(token_id, pos_id, keys, values):
    """Single-head GPT forward: token_id, pos_id -> logits."""
    tok_emb = sd3['wte'][token_id]
    pos_emb = sd3['wpe'][pos_id]
    x = [t + p for t, p in zip(tok_emb, pos_emb)]
    x = rmsnorm(x)

    # Single-head attention
    x_res = x
    x = rmsnorm(x)
    q = linear_v(x, sd3['attn_wq'])
    k = linear_v(x, sd3['attn_wk'])
    v = linear_v(x, sd3['attn_wv'])
    keys.append(k)
    values.append(v)
    # Scale by sqrt(d) where d = n_embd (single head)  ← Error #16 fix
    scale_s3 = n_embd_s3 ** 0.5
    attn_logits = [sum(q[j] * keys[t][j] for j in range(n_embd_s3)) / scale_s3
                   for t in range(len(keys))]
    attn_w = softmax_v(attn_logits)
    x_attn = [sum(attn_w[t] * values[t][j] for t in range(len(values)))
              for j in range(n_embd_s3)]
    x = linear_v(x_attn, sd3['attn_wo'])
    x = [a + b for a, b in zip(x, x_res)]   # residual after projection

    # MLP
    x_res = x
    x = rmsnorm(x)
    x = [xi.relu() for xi in linear_v(x, sd3['mlp_fc1'])]
    x = linear_v(x, sd3['mlp_fc2'])
    x = [a + b for a, b in zip(x, x_res)]

    return linear_v(x, sd3['lm_head'])

# Train Step 3: SGD
num_steps_s3 = 1000
lr_s3 = 0.1
losses_s3_log = []
for step in range(num_steps_s3):
    doc = docs_s1[step % len(docs_s1)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)
    keys_buf, vals_buf = [], []
    losses_step = []
    for i in range(n):
        tid, tgt = tokens[i], tokens[i+1]
        logits = gpt_s3(tid, i, keys_buf, vals_buf)
        probs = softmax_v(logits)
        losses_step.append(-probs[tgt].log())
    loss_s3 = (1 / n) * sum(losses_step)
    loss_s3.backward()
    lr_t = lr_s3 * (1 - step / num_steps_s3)
    for p in params_s3:
        p.data -= lr_t * p.grad
        p.grad = 0.0
    losses_s3_log.append(loss_s3.data)
    if step < 3 or (step + 1) % 200 == 0:
        print(f"step {step+1:4d}/{num_steps_s3} | loss {loss_s3.data:.4f}")


> **Step 3 checkpoint**
> - Loss should reach **~2.3** (better than Step 2's ~2.5 — attention helps!).
> - Confirm the scale factor used is $\sqrt{d} = \sqrt{n\_embd}$ in the single-head code.
> - Confirm the residual is added *after* the output projection `attn_wo`, not before.
> - Compare the attention weights for different positions in a name —
>   does "emma" attend back to the initial vowel when predicting the second 'm'?


### Step 4 — Multi-Head Attention + Adam

**New ideas:**
1. **Multi-head attention:** split the embedding into $n_{\text{head}}$ slices,
   each of dimension $d_{\text{head}} = n_{\text{embd}} / n_{\text{head}}$.
   Scale: **$\sqrt{d_{\text{head}}}$ (not $\sqrt{n_{\text{embd}}}$!)**.
   Each head can focus on different types of relationships simultaneously.

2. **Adam optimizer:** maintains per-parameter first and second gradient moments
   ($m_t, v_t$), adapting the effective learning rate for each parameter.
   This is analogous to a momentum term in molecular dynamics — the optimizer
   *remembers* the recent gradient history.

**Error #16 continued:** In this multi-head code the correct scale is
$\sqrt{d_{\text{head}}} = \sqrt{n_{\text{embd}} / n_{\text{head}}}$. Using
$\sqrt{n_{\text{embd}}}$ here would divide scores by too large a number, flattening
attention weights toward uniform — effectively disabling the attention mechanism.

**Note on GPT-2 sizes (Error #15 fix):**
GPT-2 was released in four sizes: 117M (Small), 345M (Medium), 762M (Large),
and **1542M ≈ 1.5B (XL)**. The maximum size is GPT-2 XL with ~1.5 billion
parameters, not 1.6 billion. Modern LLMs (GPT-4, Claude, Llama-3) have hundreds
of billions of parameters.


In [ ]:
# ── Step 4: Multi-head attention + Adam ──────────────────────────────────────
n_embd  = 16
n_head  = 4
n_layer = 1
block_size = 16
head_dim = n_embd // n_head    # d_head = 4

sd4 = {
    'wte':     make_mat(vocab_size, n_embd),
    'wpe':     make_mat(block_size, n_embd),
    'lm_head': make_mat(vocab_size, n_embd),
}
for i in range(n_layer):
    sd4[f'layer{i}.attn_wq'] = make_mat(n_embd, n_embd)
    sd4[f'layer{i}.attn_wk'] = make_mat(n_embd, n_embd)
    sd4[f'layer{i}.attn_wv'] = make_mat(n_embd, n_embd)
    sd4[f'layer{i}.attn_wo'] = make_mat(n_embd, n_embd)
    sd4[f'layer{i}.mlp_fc1'] = make_mat(4 * n_embd, n_embd)
    sd4[f'layer{i}.mlp_fc2'] = make_mat(n_embd, 4 * n_embd)

params_s4 = [p for mat in sd4.values() for row in mat for p in row]
print(f"num params (Step 4): {len(params_s4)}")

def gpt_s4(token_id, pos_id, keys, values):
    """Multi-head GPT forward."""
    x = [t + p for t, p in zip(sd4['wte'][token_id], sd4['wpe'][pos_id])]
    x = rmsnorm(x)
    for li in range(n_layer):
        x_res = x
        x = rmsnorm(x)
        q = linear_v(x, sd4[f'layer{li}.attn_wq'])
        k = linear_v(x, sd4[f'layer{li}.attn_wk'])
        v = linear_v(x, sd4[f'layer{li}.attn_wv'])
        keys[li].append(k)
        values[li].append(v)
        x_attn = []
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs + head_dim]
            k_h = [ki[hs:hs + head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs + head_dim] for vi in values[li]]
            # Scale by sqrt(d_head) — each head operates on head_dim dimensions  ← #16 fix
            scale_h = head_dim ** 0.5
            attn_logits = [
                sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / scale_h
                for t in range(len(k_h))
            ]
            attn_w = softmax_v(attn_logits)
            head_out = [
                sum(attn_w[t] * v_h[t][j] for t in range(len(v_h)))
                for j in range(head_dim)
            ]
            x_attn.extend(head_out)
        x = linear_v(x_attn, sd4[f'layer{li}.attn_wo'])
        x = [a + b for a, b in zip(x, x_res)]
        x_res = x
        x = rmsnorm(x)
        x = [xi.relu() for xi in linear_v(x, sd4[f'layer{li}.mlp_fc1'])]
        x = linear_v(x, sd4[f'layer{li}.mlp_fc2'])
        x = [a + b for a, b in zip(x, x_res)]
    return linear_v(x, sd4['lm_head'])

# Adam optimizer state
lr_adam = 0.01
beta1, beta2, eps_adam = 0.85, 0.99, 1e-8
m_adam = [0.0] * len(params_s4)
v_adam = [0.0] * len(params_s4)

num_steps_s4 = 1000
losses_s4_log = []
for step in range(num_steps_s4):
    doc = docs_s1[step % len(docs_s1)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)
    keys_b = [[] for _ in range(n_layer)]
    vals_b  = [[] for _ in range(n_layer)]
    losses_step = []
    for i in range(n):
        tid, tgt = tokens[i], tokens[i+1]
        logits = gpt_s4(tid, i, keys_b, vals_b)
        probs = softmax_v(logits)
        losses_step.append(-probs[tgt].log())
    loss_s4 = (1 / n) * sum(losses_step)
    loss_s4.backward()

    # Adam update
    lr_t = lr_adam * (1 - step / num_steps_s4)
    for i, p in enumerate(params_s4):
        m_adam[i] = beta1 * m_adam[i] + (1 - beta1) * p.grad
        v_adam[i] = beta2 * v_adam[i] + (1 - beta2) * p.grad ** 2
        mh = m_adam[i] / (1 - beta1 ** (step + 1))
        vh = v_adam[i] / (1 - beta2 ** (step + 1))
        p.data -= lr_t * mh / (vh ** 0.5 + eps_adam)
        p.grad = 0.0

    losses_s4_log.append(loss_s4.data)
    if step < 3 or (step + 1) % 200 == 0:
        print(f"step {step+1:4d}/{num_steps_s4} | loss {loss_s4.data:.4f}")


In [ ]:
# ── Plot learning curves for all steps ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
# Smooth with a 50-step rolling mean
def smooth(x, w=50):
    return np.convolve(x, np.ones(w)/w, mode='valid')

ax.axhline(math.log(vocab_size), color='gray', linestyle='--', label='random init baseline')
ax.plot(smooth(losses_s2_log), label='Step 2 (autograd MLP, SGD)', alpha=0.8)
ax.plot(smooth(losses_s3_log), label='Step 3 (single-head attn, SGD)', alpha=0.8)
ax.plot(smooth(losses_s4_log), label='Step 4 (multi-head attn, Adam)', alpha=0.8)
ax.set_xlabel('Training step')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('microgpt: learning curves (50-step rolling mean)')
ax.legend()
plt.tight_layout()
plt.show()
print("Each step adds a new mechanism; loss drops with each addition.")


In [ ]:
# ── Inference: sample new names from the trained model ───────────────────────
temperature = 0.5
print("\n--- Hallucinated names (Step 4 model) ---")
for sample_idx in range(20):
    keys_b = [[] for _ in range(n_layer)]
    vals_b  = [[] for _ in range(n_layer)]
    token_id = BOS
    sample = []
    for pos_id in range(block_size):
        logits = gpt_s4(token_id, pos_id, keys_b, vals_b)
        probs = softmax_v([l / temperature for l in logits])
        token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
        if token_id == BOS:
            break
        sample.append(uchars[token_id])
    print(f"  sample {sample_idx+1:2d}: {''.join(sample)}")


> **Step 4 checkpoint**
> - With `n_layer=1`: loss ~2.3.
> - Try `n_layer=2` and re-run: expect ~2.1–2.2.
> - Confirm each head uses `head_dim**0.5` as scale, **not** `n_embd**0.5`.
> - Adam converges faster than SGD (compare step 4 vs step 3 learning curves).


---
## Physics Payoff — Attention as a Data-Dependent Many-Body Kernel

So far we used the Transformer on a language task.
Now we interpret the attention mechanism through a physics lens and apply it to a
toy spin-chain problem.

### Attention ~ dynamic pairwise interactions

In a physical system with $N$ particles (or spins), a *fixed* pairwise kernel
$K_{ij}$ computes a weighted sum of contributions from all neighbours:

$$x_i^{\text{out}} = \sum_j K_{ij} \, v_j$$

**SchNet** (L11) uses a distance-based radial cutoff: $K_{ij}$ depends only on
$|r_i - r_j|$, and neighbours are fixed by the cutoff sphere.

**Transformer attention** uses a *data-dependent* kernel:

$$A_{ij} = \frac{\exp\left(q_i \cdot k_j / \sqrt{d}\right)}{\sum_{j'} \exp\left(q_i \cdot k_{j'} / \sqrt{d}\right)}, \qquad x_i^{\text{out}} = \sum_j A_{ij} \, v_j$$

The "neighbourhood" for token $i$ is selected *dynamically* based on the content
of the sequence — not a fixed geometric cutoff. This makes Transformers
**soft, adaptive, content-based** many-body models.

| Method | Pairwise kernel | Neighbourhood |
|--------|----------------|---------------|
| SchNet (L11) | radial basis + cutoff | static (geometry only) |
| Transformer (here) | $\text{softmax}(QK^\top / \sqrt{d})$ | dynamic (content-dependent) |

**Forward-pointer to L11:** When we study GNNs next, we will formalize
*message passing* and see that SchNet is a special case of attention with
a fixed (non-learned) kernel. The Transformer relaxes this to a learned,
input-dependent kernel.


In [ ]:
# ── Visualize attention matrix on a toy Ising spin chain ─────────────────────
#
# Setup: N=8 Ising spins in a random config. We feed the spin sequence as tokens
# into a *randomly initialized* single-layer Transformer and inspect the raw
# attention weights A[t, s] = "how much does position t attend to position s?"
#
# This is a diagnostic, not a trained model. The point is to see the *structure*
# of attention (causal, all positions) before any learning.

import numpy as np
import matplotlib.pyplot as plt

N_SPINS = 8
n_embd_phy = 16
n_head_phy = 4
head_dim_phy = n_embd_phy // n_head_phy

# Spin states: +1 (up) or -1 (down) — encode as token ids 0 (+1) and 1 (-1)
SPIN_VOCAB = 2
spin_config = np.random.choice([0, 1], size=N_SPINS)  # 0 = up, 1 = down
print(f"Spin configuration (0=up, 1=down): {spin_config.tolist()}")

# Random parameters for a single-layer transformer (numpy only, for visualization)
rng = np.random.default_rng(42)
wte_phy = rng.normal(0, 0.1, (SPIN_VOCAB, n_embd_phy))   # token embed
wpe_phy = rng.normal(0, 0.1, (N_SPINS,   n_embd_phy))   # position embed
Wq = rng.normal(0, 0.1, (n_embd_phy, n_embd_phy))
Wk = rng.normal(0, 0.1, (n_embd_phy, n_embd_phy))

# Build embeddings
X = wte_phy[spin_config] + wpe_phy        # shape (N_SPINS, n_embd)

# Compute Q, K
Q = X @ Wq                                # (N_SPINS, n_embd)
K = X @ Wk                                # (N_SPINS, n_embd)

# Compute full attention matrix (all positions, no causal mask — for visualization)
# Shape: (N_SPINS, N_SPINS)
A_full = np.zeros((N_SPINS, N_SPINS))
for t in range(N_SPINS):
    scores = Q[t] @ K.T / np.sqrt(n_embd_phy)  # use n_embd (single-head analog)
    scores_exp = np.exp(scores - scores.max())
    A_full[t] = scores_exp / scores_exp.sum()

print(f"Attention matrix shape: {A_full.shape}")
print(f"Row sums (should all be 1.0): {A_full.sum(axis=1).round(4).tolist()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: full attention matrix
im = axes[0].imshow(A_full, cmap='hot', vmin=0, vmax=A_full.max())
axes[0].set_xlabel("Source position $s$")
axes[0].set_ylabel("Query position $t$")
axes[0].set_title("Full attention weights $A_{ts}$\n(no causal mask; random init)")
axes[0].set_xticks(range(N_SPINS))
axes[0].set_yticks(range(N_SPINS))
spin_labels = ['↑' if s == 0 else '↓' for s in spin_config]
axes[0].set_xticklabels(spin_labels)
axes[0].set_yticklabels(spin_labels)
plt.colorbar(im, ax=axes[0])

# Right: causal (autoregressive) attention — mask future positions
causal_mask = np.tril(np.ones((N_SPINS, N_SPINS)))
A_causal = np.zeros((N_SPINS, N_SPINS))
for t in range(N_SPINS):
    scores = Q[t] @ K.T / np.sqrt(n_embd_phy)
    scores_masked = scores.copy()
    scores_masked[t+1:] = -1e9   # mask future
    scores_exp = np.exp(scores_masked - scores_masked[:t+1].max())
    scores_exp[t+1:] = 0.0
    A_causal[t, :t+1] = scores_exp[:t+1] / scores_exp[:t+1].sum()

im2 = axes[1].imshow(A_causal, cmap='hot', vmin=0, vmax=A_causal.max())
axes[1].set_xlabel("Source position $s$")
axes[1].set_ylabel("Query position $t$")
axes[1].set_title("Causal attention weights $A_{ts}$\n(autoregressive, random init)")
axes[1].set_xticks(range(N_SPINS))
axes[1].set_yticks(range(N_SPINS))
axes[1].set_xticklabels(spin_labels)
axes[1].set_yticklabels(spin_labels)
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()


In [ ]:
# ── Interpretive summary: dynamic vs static neighbourhood ────────────────────
#
# Key observation: A[t, s] depends on the *content* of positions t and s
# (via Q[t] and K[s]), not just their geometric distance.
#
# This is the defining difference between attention-based models and fixed-kernel
# message-passing networks like SchNet.

print("=== Dynamic vs Static Neighbourhood ===")
print()
print("SchNet (L11, GNN):")
print("  Neighbours of atom i: all j with |r_i - r_j| < r_cut  (fixed by geometry)")
print("  K_{ij} = radial_basis(|r_i - r_j|) * cutoff function   (not learned here)")
print()
print("Transformer (this lecture):")
print("  A[t, s] = softmax(q_t · k_s / sqrt(d))[s]")
print("          → depends on the *values* at positions t and s")
print("          → same spin config but different learned W_Q, W_K → different A")
print()
print("Summary:")
print("  Transformer attention is a DATA-DEPENDENT many-body kernel.")
print("  The model dynamically selects which past positions are 'neighbours'")
print("  based on input content — not fixed by geometry or distance.")
print()
print("Forward pointer to L11:")
print("  GNNs formalize 'message passing'; SchNet is attention with a fixed,")
print("  non-learned radial kernel. Transformer relaxes this to a learned,")
print("  input-dependent kernel. Both are special cases of the same blueprint.")


> **Physics payoff exercise**
> 1. Run the attention visualization above on the spin chain.
> 2. Pick the query position corresponding to the 3rd spin (index 2).
>    Which source positions receive the highest attention weight?
> 3. Now flip one spin and re-run. Does the attention pattern change?
>    (It should — the kernel is *data-dependent*.)
> 4. In one sentence, contrast this with SchNet's neighbour selection (L11).


---
## Session B — Evaluation, Validation, and Data Pipeline
### (Term Project Preparation)

This session is a structured walkthrough of the key evaluation practices that
your term project grade depends on. Work through each checkpoint and apply it
to your own project dataset.

Topics covered:
1. Train / validation / test split design
2. Learning curve interpretation (overfitting diagnosis)
3. Data leakage — common pitfalls and checks
4. Baseline comparisons and result tables
5. Reporting checklist for the term project


### B1 — Train / Validation / Test Split Design

**Why three splits?**
- **Train:** the model *learns* from this data.
- **Validation:** used during development to tune hyperparameters, select models,
  and detect overfitting. You are allowed to look at validation performance
  as often as you like.
- **Test:** held out completely until final evaluation. Looking at test
  performance repeatedly and making decisions based on it effectively leaks
  information and inflates reported accuracy.

**Physics analogy:** In spectroscopy you calibrate the instrument on known
peaks (train), adjust settings on a second calibration sample (val), then
measure your unknown sample (test) exactly once.

**Split sizes:** A common rule of thumb is 70 / 15 / 15 or 80 / 10 / 10.
For small datasets (< 1000 samples), consider k-fold cross-validation on train+val.

**Stratification:** If your data has classes (e.g. phase labels, chemical species),
stratify the split so each split has the same class proportions as the full dataset.


In [ ]:
# ── B1: Illustrate train/val/test split ──────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Synthetic physics dataset: regression of a noisy Morse potential
N_DATA = 500
r = np.linspace(0.8, 5.0, N_DATA)

# Morse potential: V(r) = D_e * (1 - exp(-a*(r - r_e)))^2
D_e, a, r_e = 2.5, 1.8, 1.5
V_morse = D_e * (1 - np.exp(-a * (r - r_e)))**2
noise = np.random.normal(0, 0.05, N_DATA)
y = V_morse + noise

# Split: 70 / 15 / 15
idx = np.random.permutation(N_DATA)
n_train = int(0.70 * N_DATA)
n_val   = int(0.15 * N_DATA)
train_idx = idx[:n_train]
val_idx   = idx[n_train:n_train + n_val]
test_idx  = idx[n_train + n_val:]

r_train, y_train = r[train_idx], y[train_idx]
r_val,   y_val   = r[val_idx],   y[val_idx]
r_test,  y_test  = r[test_idx],  y[test_idx]

print(f"Dataset size : {N_DATA}")
print(f"Train        : {len(train_idx)} ({len(train_idx)/N_DATA:.0%})")
print(f"Validation   : {len(val_idx)}  ({len(val_idx)/N_DATA:.0%})")
print(f"Test         : {len(test_idx)}  ({len(test_idx)/N_DATA:.0%})")

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(r_train, y_train, s=4, alpha=0.5, label='Train', color='steelblue')
ax.scatter(r_val,   y_val,   s=4, alpha=0.5, label='Validation', color='orange')
ax.scatter(r_test,  y_test,  s=4, alpha=0.5, label='Test', color='green')
ax.plot(r, V_morse, 'k--', lw=1, label='True Morse V(r)', alpha=0.6)
ax.set_xlabel('r (Å)')
ax.set_ylabel('V(r) (eV)')
ax.set_title('Train/Val/Test split on Morse potential data')
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()


### B2 — Learning Curves and Overfitting Diagnosis

A *learning curve* plots training loss and validation loss vs training step
(or epoch). Reading the curve:

| Pattern | Diagnosis |
|---------|-----------|
| Both losses decrease and converge | Good — model is learning |
| Train loss low, val loss high and diverging | **Overfitting** — model memorizes train data |
| Both losses high and flat | Underfitting — model too simple or learning rate too low |
| Val loss starts rising after early steps | Overfitting begins — this is the stopping point |

Early stopping: save the model checkpoint at the epoch with minimum validation loss.


In [ ]:
# ── B2: Simulate train/val learning curves ────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
steps = np.arange(1, 201)

# Simulate a realistic train/val loss trajectory
train_loss = 2.5 * np.exp(-0.025 * steps) + 0.05 + 0.02 * np.random.randn(len(steps))
# Val loss: initially tracks train, then starts to diverge at ~step 80
val_loss = (2.5 * np.exp(-0.025 * steps) + 0.12
            + 0.3 * np.maximum(0, (steps - 80) / 100) ** 2
            + 0.03 * np.random.randn(len(steps)))

best_step = steps[np.argmin(val_loss)]
best_val  = val_loss.min()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, train_loss, label='Train loss', color='steelblue')
ax.plot(steps, val_loss,   label='Val loss',   color='orange')
ax.axvline(best_step, color='red', linestyle='--', alpha=0.7,
           label=f'Best val step = {best_step}')
ax.scatter([best_step], [best_val], color='red', zorder=5)
ax.set_xlabel('Training step')
ax.set_ylabel('Loss')
ax.set_title('Learning curves: overfitting begins around step 80')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Best validation loss = {best_val:.4f} at step {best_step}")
print("Use early stopping: save checkpoint at minimum val loss.")


### B3 — Data Leakage: Common Pitfalls

**Data leakage** occurs when information from the test (or validation) set
inadvertently flows into the training process. It is the single most common
cause of over-optimistic results in ML papers.

**Common sources:**
1. **Normalization with full-dataset statistics:** computing mean/std over all data
   (including test) before splitting. Fix: fit normalizer on train, apply to val/test.
2. **Correlated samples across splits:** e.g. in molecular datasets, two conformers
   of the same molecule may end up in train and test. Fix: split by molecule, not conformation.
3. **Temporal leakage:** using future data to predict the past (time-series).
   Fix: always split chronologically.
4. **Target engineering based on all data:** e.g. normalizing the energy target
   by the global mean. Fix: use train-set mean only.


In [ ]:
# ── B3: Data leakage demonstration ───────────────────────────────────────────
import numpy as np

np.random.seed(42)
N = 200
X_all = np.random.randn(N, 4)         # 4 features
y_all = X_all[:, 0] * 2 + np.random.randn(N) * 0.5   # y depends on feature 0

# ---- WRONG: normalize using all data ----------------------------------------
mean_wrong = X_all.mean(axis=0)
std_wrong  = X_all.std(axis=0) + 1e-8
X_norm_wrong = (X_all - mean_wrong) / std_wrong

# Split after normalization (information from test used in normalization)
X_train_wrong = X_norm_wrong[:160]
X_test_wrong  = X_norm_wrong[160:]

# ---- CORRECT: normalize using only train data --------------------------------
X_train_raw = X_all[:160]
X_test_raw  = X_all[160:]
mean_correct = X_train_raw.mean(axis=0)
std_correct  = X_train_raw.std(axis=0) + 1e-8
X_train_correct = (X_train_raw - mean_correct) / std_correct
X_test_correct  = (X_test_raw  - mean_correct) / std_correct   # use TRAIN stats

print("Feature 0 statistics:")
print(f"  Train mean  (train-stats): {X_train_correct[:, 0].mean():.4f}  (expect ~0)")
print(f"  Test  mean  (train-stats): {X_test_correct[:, 0].mean():.4f}   (may differ from 0)")
print()
print(f"  Train mean  (all-stats):   {X_train_wrong[:, 0].mean():.4f}")
print(f"  Test  mean  (all-stats):   {X_test_wrong[:, 0].mean():.4f}  (artificially 0 — leakage!)")
print()
print("Correct approach: fit normalizer on TRAIN, transform both train AND test.")


### B4 — Baseline Comparisons and Result Tables

Every ML result needs a **baseline**: the simplest model against which to measure
improvement. Without a baseline, a reported loss of 0.35 is meaningless.

**Standard baselines:**
- **Constant predictor:** always predict the mean (for regression) or majority class.
- **Linear model:** ridge regression or logistic regression — fast to train.
- **Previous work:** the best published method on the same benchmark.

**Reporting checklist:**
- Report mean ± std over multiple random seeds (at least 3).
- Specify the split sizes and random seed.
- Specify all relevant hyperparameters.
- Use test set exactly once (after all tuning is done on val).


In [ ]:
# ── B4: Build a minimal comparison table ─────────────────────────────────────
import numpy as np

np.random.seed(42)

# Simulate results from 3 random seeds for 3 methods
methods = {
    'Constant (mean)': [0.248, 0.251, 0.249],    # near Var(y)
    'Linear regression': [0.132, 0.128, 0.135],
    'Your model (MLP)': [0.087, 0.091, 0.083],
}

print(f"{'Method':<25}  {'MAE mean':>10}  {'MAE std':>10}  {'vs constant':>12}")
print("-" * 65)
constant_mean = np.mean(methods['Constant (mean)'])
for name, vals in methods.items():
    m = np.mean(vals)
    s = np.std(vals)
    vs = (constant_mean - m) / constant_mean * 100
    print(f"{name:<25}  {m:>10.4f}  {s:>10.4f}  {vs:>+11.1f}%")

print()
print("Your model reduces MAE by ~65% vs the constant-mean baseline.")
print("Linear regression reduces it by ~47% — a non-trivial intermediate.")
print("Always report the linear-model baseline; it is often surprisingly strong.")


> **Session B checkpoint (Term Project)**
>
> For your own term project dataset, complete the following:
>
> 1. **Split design:** Specify train/val/test sizes and justify the ratio.
>    Is stratification needed? Are there correlated samples that must stay together?
>
> 2. **Learning curve:** Train your baseline model and plot train + val loss vs step.
>    Identify the epoch with minimum val loss. Does it show overfitting?
>
> 3. **Leakage check:** Confirm that your normalizer (or any preprocessing)
>    is fitted on train data only.
>
> 4. **Baseline table:** Fill in the table below for your project:
>    | Method | Metric (mean ± std, N=3 seeds) |
>    |--------|-------------------------------|
>    | Constant predictor | |
>    | Linear model | |
>    | Your proposed model | |


---
## Appendix — Optional: Fine-Tuning GPT-2 on a Physics Corpus

This appendix shows how to load a **pretrained GPT-2 (124M)** from HuggingFace
and fine-tune it on a small physics text corpus (arXiv abstract snippets).
This bridges the "build from scratch" understanding above to real-world model usage.

**Requirements:** Google Colab T4 GPU. Expected runtime: ~10 minutes.
Uncomment the pip install in the setup cell above before running.

**Why fine-tune, not train from scratch?**
GPT-2 was pretrained on ~40GB of web text and already understands English grammar,
scientific notation, and common physics terminology. Fine-tuning on 10k physics
sentences takes a few minutes and produces coherent physics text.
Training from scratch on the same data would take days on a single GPU and
produce much worse results.

**Physics perspective:** This is analogous to *transfer learning* in material science:
start from a foundation model (MACE-MP-0, trained on all of Materials Project),
fine-tune on your specific material system (L13). The same paradigm — pretrain on
broad data, adapt to narrow domain — applies to language models.


In [ ]:
# ── Appendix: HuggingFace GPT-2 fine-tune (needs GPU + transformers) ─────────
# Uncomment ALL lines below to run this appendix.

# import torch
# from transformers import (GPT2LMHeadModel, GPT2Tokenizer,
#                            TextDataset, DataCollatorForLanguageModeling,
#                            Trainer, TrainingArguments)
# import os, textwrap

# # ── 1. Load pretrained GPT-2 (124M) ─────────────────────────────────────────
# model_name = "gpt2"   # 124M parameters (GPT-2 Small)
# tokenizer = GPT2Tokenizer.from_pretrained(model_name)
# model     = GPT2LMHeadModel.from_pretrained(model_name)
# print(f"GPT-2 Small: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")
# # GPT-2 sizes: Small 117M, Medium 345M, Large 762M, XL ~1.5B (1542M)  ← #15 fix

# # ── 2. Toy physics corpus (replace with arXiv abstracts for real training) ───
# PHYSICS_TEXT = textwrap.dedent("""
#     The quantum harmonic oscillator is a fundamental model in quantum mechanics.
#     Its energy eigenvalues are equally spaced: E_n = hbar * omega * (n + 1/2).
#     The ground state wavefunction is a Gaussian centered at the equilibrium position.
#     Neural networks can approximate solutions to the Schrodinger equation.
#     Machine learning potentials learn the Born-Oppenheimer potential energy surface.
#     Graph neural networks respect the permutation symmetry of atoms.
#     Normalizing flows model probability distributions with invertible transformations.
#     The Boltzmann distribution describes thermal equilibrium in statistical mechanics.
#     Diffusion models generate samples by reversing a stochastic noise process.
#     Variational autoencoders compress data into a low-dimensional latent space.
#     The attention mechanism allows transformers to model long-range dependencies.
# """).strip()

# # Save corpus to file (fine-tuning API expects a text file)
# corpus_path = "/tmp/physics_corpus.txt"
# with open(corpus_path, "w") as f:
#     # Replicate to get enough tokens for a demo run
#     f.write((PHYSICS_TEXT + "\n") * 200)
# print(f"Corpus: {len(open(corpus_path).read().split())} words")

# # ── 3. Tokenize ──────────────────────────────────────────────────────────────
# tokenizer.pad_token = tokenizer.eos_token
# dataset = TextDataset(tokenizer=tokenizer, file_path=corpus_path, block_size=64)
# collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
# print(f"Dataset: {len(dataset)} chunks of 64 tokens")

# # ── 4. Fine-tune ─────────────────────────────────────────────────────────────
# training_args = TrainingArguments(
#     output_dir="/tmp/gpt2-physics",
#     overwrite_output_dir=True,
#     num_train_epochs=3,
#     per_device_train_batch_size=8,
#     save_steps=50,
#     save_total_limit=1,
#     logging_steps=10,
#     learning_rate=5e-5,
#     warmup_steps=20,
#     fp16=torch.cuda.is_available(),
#     report_to="none",
# )
# trainer = Trainer(
#     model=model, args=training_args,
#     data_collator=collator, train_dataset=dataset,
# )
# trainer.train()
# print("Fine-tuning complete.")

# # ── 5. Generate physics text ──────────────────────────────────────────────────
# inputs = tokenizer("The variational free energy is", return_tensors="pt")
# outputs = model.generate(
#     **inputs, max_new_tokens=60, do_sample=True,
#     temperature=0.7, top_p=0.9,
#     pad_token_id=tokenizer.eos_token_id,
# )
# generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print("\nGenerated text (fine-tuned GPT-2):")
# print(generated)


---
## Summary

### Session A — microgpt in 4 steps

| Step | Addition | Final loss |
|------|---------|-----------|
| 0 (baseline) | Bigram count (no neural net) | ~2.9 bits/char |
| 1 | MLP + SGD (manual) | ~2.7 |
| 2 | Autograd (`Value`) | same architecture, easier code |
| 3 | Single-head attention + position embeddings | ~2.3 |
| 4 | Multi-head attention + Adam | ~2.2 (or lower with n_layer=2) |

**Key error fixes in v2:**
- **#15:** GPT-2 *XL* has ~1.5B (1542M) parameters, not 1.6B.
- **#16:** Attention scale is $\sqrt{d}$ where $d$ = feature dimension *of that head*:
  $\sqrt{n\_embd}$ for single-head, $\sqrt{n\_embd / n\_head}$ for each head in multi-head.

### Session B — Evaluation / Data Pipeline

- Always split *before* fitting any normalizer or preprocessing.
- Plot train + val loss curves; use early stopping at minimum val loss.
- Baseline table: constant predictor → linear model → your model.
- Check for data leakage (correlated samples, temporal ordering, normalizer fit).

### Physics payoff

Transformer attention $A_{ij} = \text{softmax}(q_i \cdot k_j / \sqrt{d})$ is a
**data-dependent many-body kernel**: the "effective neighbourhood" of token $i$
is selected dynamically based on content, unlike the fixed radial cutoff of SchNet.
This distinction becomes central when we study GNNs in L11.

### Optional appendix

HuggingFace GPT-2 fine-tuning: load a 124M pretrained model, adapt it to a
physics corpus in ~10 minutes on Colab T4 — the same transfer-learning paradigm
as MACE-MP-0 foundation model fine-tuning (L13).
